### Silver — nettoyage et fiabilisation des ventes

**Rôle de ce notebook :**
- Lire `bronze.ventes.ventes` (miroir fidèle de la source, non modifié)
- Appliquer les règles de qualité de `libs/data_quality.py`
- Séparer les lignes valides des lignes rejetées, avec le motif exact du rejet
- Écrire deux tables silver : une propre, une de quarantaine

**Toujours à la même granularité que bronze** : une ligne = une vente. Aucune agrégation, aucune jointure de dimension ici — ça reste le rôle de gold.


In [0]:
%run "../libs/data_quality"

In [0]:
from pyspark.sql import functions as F

In [0]:
df_bronze = spark.table("bronze.ventes.ventes")
print(f"Lignes en entrée (bronze) : {df_bronze.count()}")

In [0]:
# Règle 1 — Standardisation de la casse (région)
# Ne rejette rien, corrige seulement. Voir `data_quality.py` pour le détail.

df = standardize_text_column(df_bronze, "region")

In [0]:
# Règle 2 — Cast sécurisé de la date
# Toute ligne dont la date ne peut pas être castée part en quarantaine.

df, rejets_date = safe_cast_date(df, "date_vente")
log_quality_summary("Cast date_vente", df, rejets_date)

In [0]:
# Règle 3 — Quantités invalides

# Négatives, nulles ou à zéro : rejetées, elles fausseraient toute agrégation
# de chiffre d'affaires en gold.

df, rejets_quantite = filter_invalid_quantities(df)
log_quality_summary("Filtrage quantite", df, rejets_quantite)

In [0]:
# Règle 4 — Champs critiques manquants

#`produit` et `prix_unitaire` sont indispensables pour toute agrégation de
# CA ou toute jointure avec une dimension produit en gold. Contrairement aux
# règles précédentes, celle-ci est assez simple (un `isNull` direct) pour ne
# pas justifier sa propre fonction dans `data_quality.py` — on l'écrit
# directement ici. Si une 3ème règle de ce type apparaissait, ce serait le
# signal qu'il faut la généraliser dans la librairie.

champs_critiques_manquants = F.col("produit").isNull() | F.col("prix_unitaire").isNull()

rejets_champs_manquants = df.filter(champs_critiques_manquants)
df = df.filter(~champs_critiques_manquants)

log_quality_summary("Champs critiques manquants", df, rejets_champs_manquants)

In [0]:
# Règle 5 — Doublons exacts

# Éliminés silencieusement (ce ne sont pas des erreurs à auditer, juste des
# répétitions), donc pas de DataFrame de rejet ici.

n_avant = df.count()
df = remove_exact_duplicates(df)
n_apres = df.count()
print(f"Doublons supprimés : {n_avant - n_apres}")

In [0]:
# Assemblage de la table de rejets consolidée
#  
# Chaque étape produit un DataFrame de rejet avec un schéma légèrement
# différent (le cast de date change le type de `date_vente` en cours de
# route). On normalise chaque lot sur un socle commun de colonnes avant de
# les union, avec un motif explicite — c'est ce qui rendra la table de
# quarantaine interrogeable simplement, sans avoir à connaître l'ordre des
# règles qui l'ont produite.

colonnes_rejet = ["id_vente", "region", "date_vente", "quantite", "produit", "prix_unitaire"]

def normaliser_rejet(df_rejet, motif: str):
    return (
        df_rejet
        .withColumn("date_vente", F.col("date_vente").cast("string"))
        .select(*colonnes_rejet)
        .withColumn("motif_rejet", F.lit(motif))
    )

df_rejets = (
    normaliser_rejet(rejets_date, "date_invalide")
    .unionByName(normaliser_rejet(rejets_quantite, "quantite_invalide"))
    .unionByName(normaliser_rejet(rejets_champs_manquants, "champ_critique_manquant"))
    .withColumn("_rejected_at", F.current_timestamp())
)

print(f"Total lignes en quarantaine : {df_rejets.count()}")
display(df_rejets.groupBy("motif_rejet").count())

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.ventes.ventes_clean")
)

(
    df_rejets.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.ventes.ventes_rejected")
)

print(f"silver.ventes.ventes_clean : {df.count()} lignes")
print(f"silver.ventes.ventes_rejected : {df_rejets.count()} lignes")

In [0]:
%sql
SELECT * FROM silver.ventes.ventes_clean LIMIT 5;

In [0]:
%sql
SELECT motif_rejet, COUNT(*) AS nb
FROM silver.ventes.ventes_rejected
GROUP BY motif_rejet;